In [66]:
import yfinance as yf
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

In [67]:
# Load data from Yahoo Finance API
data = {
    'AAPL': 'AAPL',
    'GOOGL': 'GOOG',
}

In [68]:
class StockDataset(Dataset):
    def __init__(self, df, seq_len):
        self.df = df
        self.seq_len = seq_len
        if seq_len <= 0:
            raise ValueError("Sequence length must be positive")

    def __len__(self):
        return len(self.df) - self.seq_len + 1

    def __getitem__(self, idx):
        sequence = self.df.iloc[idx:idx + self.seq_len]
        target = self.df.iloc[idx + self.seq_len]['Close']
        return {
            'sequence': torch.tensor(sequence.values),
            'target': torch.tensor(target)
        }

In [69]:
class SwingTradingModel(torch.nn.Module):
    def __init__(self):
        super(SwingTradingModel, self).__init__()
        self.fc1 = torch.nn.Linear(8, 16)  # changed from int to float
        self.fc2 = torch.nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [70]:
def prepare_data(df, seq_len):
    if seq_len <= 0:
        raise ValueError("Sequence length must be positive")
    
    dataset = StockDataset(df, seq_len)
    return dataset

In [71]:
def train_model(model, device, dataloader, epochs):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        for sequence, target in dataloader:
            optimizer.zero_grad()
            output = model(sequence.view(-1, 8))  # changed from torch.relu to just output
            loss_fn = torch.nn.MSELoss()
            loss = loss_fn(output.view(-1), target)
            loss.backward()
            optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {loss.item()}')

In [72]:
def evaluate_model(model, device, dataloader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in dataloader:
            sequence, target = batch['sequence'], batch['target']
            sequence, target = sequence.to(device), target.to(device)

            # Forward pass and compute loss
            outputs = model(sequence)
            loss_fn = torch.nn.MSELoss()
            loss = loss_fn(outputs.view(-1), target)
            losses.append(loss.item())

    return np.mean(losses)

In [73]:
# Load data
df = yf.download('AAPL', start='2010-01-01', end='2022-02-26')

[*********************100%%**********************]  1 of 1 completed


In [74]:
# Preprocess data
scaler = MinMaxScaler()
df[['Open', 'High', 'Low', 'Close']] = scaler.fit_transform(df[['Open', 'High', 'Low', 'Close']])

In [75]:
# Prepare dataset and dataloader
seq_len = 30
dataloader = prepare_data(df, seq_len)

In [76]:
# Create model and move to device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SwingTradingModel()
model.to(device)

SwingTradingModel(
  (fc1): Linear(in_features=8, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=1, bias=True)
)

In [77]:
# Train model
epochs = 1000
train_model(model, device, dataloader, epochs)

AttributeError: 'str' object has no attribute 'view'

In [18]:
# Evaluate model
loss = evaluate_model(model, device, dataloader)
print(f'Test Loss: {loss}')

ZeroDivisionError: division by zero